In [ ]:
# Install system dependencies
!apt-get update
!apt-get install -y poppler-utils

# Install Python packages
!pip install torch torchvision transformers accelerate safetensors einops addict easydict
!pip install pdf2image Pillow opencv-python numpy tqdm

# Install specific versions for DeepSeek OCR compatibility
!pip uninstall -y transformers
!pip install transformers==4.46.3 tokenizers==0.20.3
!pip install flash-attn==2.7.3 --no-build-isolation

print("✅ All dependencies installed!")
print("⚠️ Click 'Runtime' → 'Restart Runtime' to apply changes, then run the cells below.")

In [ ]:
# OPTION 1: Mount Google Drive (Recommended for many files)
# This allows you to access your PDFs directly from your Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted!")
except ImportError:
    print("⚠️ Not running in Colab or Google Drive not available.")

In [ ]:
# OPTION 2: Upload local files directly
# Run this cell if you prefer to upload files from your computer now.
# A button will appear to select files.

try:
    from google.colab import files
    import shutil
    from pathlib import Path

    print("👇 Click below to upload your local PDF files:")
    uploaded = files.upload()

    if uploaded:
        # Create a directory for uploaded files
        upload_dir = Path('/content/uploaded_pdfs')
        upload_dir.mkdir(exist_ok=True)

        for filename in uploaded.keys():
            destination = upload_dir / filename
            shutil.move(filename, destination)
        
        print(f"\n✅ Uploaded {len(uploaded)} files to: {upload_dir}")
        print(f"👉 Please update CONFIG below path to: '{upload_dir}'")
    else:
        print("No files uploaded.")

except ImportError:
    print("⚠️ This feature only works in Google Colab.")
except Exception as e:
    print(f"Upload error: {e}")

In [ ]:
"""
DeepSeek OCR Scraper - Google Colab Version
Extract text from PDFs using DeepSeek OCR on GPU.
"""

import os
import time
from pathlib import Path
from typing import List, Optional

# Image processing
from transformers import AutoModel, AutoTokenizer
import torch
import numpy as np
from PIL import Image
from pdf2image import convert_from_path
from tqdm import tqdm

# ============================================
# CONFIGURATION
# ============================================

CONFIG = {
    # Paths
    # Option A (Drive): '/content/drive/MyDrive/Thesis/files/2'
    # Option B (Upload): '/content/uploaded_pdfs'
    'original_pdfs_dir': '/content/drive/MyDrive/Thesis/files/2',
    
    # Output Directory (will be created)
    'ocr_output_dir': '/content/ocr_output',

    # PDF Processing
    'dpi': 400,

    # DeepSeek Transformers settings
    'deepseek_model': 'deepseek-ai/DeepSeek-OCR',
    'deepseek_prompt': '<image>\nFree OCR.',
    'deepseek_base_size': 1024,
    'deepseek_image_size': 640,
}

# Verify paths exist
if not Path(CONFIG['original_pdfs_dir']).exists():
    print(f"⚠️ WARNING: Input directory does not exist: {CONFIG['original_pdfs_dir']}")
    print("Check the path in CONFIG. If using 'Upload', ensure you ran the upload cell above and updated the path.")
else:
    print(f"✅ Input directory found: {CONFIG['original_pdfs_dir']}")


In [ ]:
class DeepSeekOCRScraper:
    """
    Scraper to extract text from PDFs using DeepSeek OCR
    """

    def __init__(self):
        """Initialize the OCR scraper"""

        # Setup paths
        self.original_pdfs_dir = Path(CONFIG['original_pdfs_dir'])
        self.ocr_output_dir = Path(CONFIG['ocr_output_dir'])
        self.ocr_output_dir.mkdir(exist_ok=True, parents=True)

        # Load DeepSeek Transformers model
        self._load_deepseek_transformers()

    def _load_deepseek_transformers(self):
        """Load DeepSeek-OCR model using Transformers"""
        try:
            print("Loading DeepSeek-OCR with Transformers (first time downloads ~6.7GB)...")

            self.deepseek_tokenizer = AutoTokenizer.from_pretrained(
                CONFIG['deepseek_model'],
                trust_remote_code=True
            )

            self.deepseek_model = AutoModel.from_pretrained(
                CONFIG['deepseek_model'],
                trust_remote_code=True,
                use_safetensors=True
            )

            # Move to GPU if available
            if torch.cuda.is_available():
                print(f"  ✓ CUDA available (GPU: {torch.cuda.get_device_name(0)})")
                self.deepseek_model = self.deepseek_model.eval().cuda().to(torch.bfloat16)
            else:
                print("  ⚠ CUDA not available, using CPU (will be slow)")
                self.deepseek_model = self.deepseek_model.eval()

            print("  ✓ DeepSeek-OCR loaded successfully")

        except Exception as e:
            print(f"  ✗ Error loading DeepSeek: {e}")
            self.deepseek_model = None
            self.deepseek_tokenizer = None
            raise e

    def ocr_deepseek(self, image: Image.Image) -> str:
        """
        Run DeepSeek OCR using Transformers
        """
        if not hasattr(self, 'deepseek_model') or self.deepseek_model is None:
            return "ERROR: DeepSeek model not loaded"

        try:
            # Save image temporarily (model.infer requires file path)
            temp_image_path = self.ocr_output_dir / "temp_deepseek_image.png"
            image.save(temp_image_path, format='PNG')

            # Run DeepSeek inference using model.infer()
            result = self.deepseek_model.infer(
                self.deepseek_tokenizer,
                prompt=CONFIG['deepseek_prompt'],
                image_file=str(temp_image_path),
                output_path=str(self.ocr_output_dir),
                base_size=CONFIG['deepseek_base_size'],
                image_size=CONFIG['deepseek_image_size'],
                crop_mode=True,
                save_results=False,
                test_compress=True
            )

            # Clean up temp file
            if temp_image_path.exists():
                temp_image_path.unlink()

            return result if result else ""

        except Exception as e:
            print(f"DeepSeek error: {e}")
            return f"ERROR: {str(e)}"

    def process_pdf(self, pdf_path: Path) -> str:
        """
        Process a PDF file and extract text
        """
        # Convert PDF to images
        try:
            images = convert_from_path(str(pdf_path), dpi=CONFIG['dpi'])
        except Exception as e:
            print(f"PDF conversion error for {pdf_path.name}: {e}")
            return ""

        all_text_parts = []

        # Process each page
        for page_num, image in enumerate(images, 1):
            try:
                page_text = self.ocr_deepseek(image)
                all_text_parts.append(page_text)
                print(f"    Page {page_num}/{len(images)} done")

            except Exception as e:
                print(f"    Error processing page {page_num}: {e}")
                all_text_parts.append(f"[ERROR ON PAGE {page_num}]")

        # Combine all pages
        full_text = "\n\n".join(all_text_parts)
        return full_text

    def run_scraper(self):
        """Run the scraping process for all PDFs"""
        print("="*60)
        print("DEEPSEEK OCR SCRAPER")
        print("="*60)

        # Get PDF files
        pdf_files = sorted(list(self.original_pdfs_dir.glob('*.pdf')))
        print(f"Found {len(pdf_files)} PDF files in {self.original_pdfs_dir}")

        for pdf_file in tqdm(pdf_files, desc="Processing PDFs", unit="file"):
            output_file = self.ocr_output_dir / f"{pdf_file.stem}.txt"
            
            # Skip if already exists
            if output_file.exists():
                print(f"\n  ✓ Skipping {pdf_file.name} (already processed)")
                continue

            print(f"\n  Processing {pdf_file.name}...")
            start_time = time.time()
            
            text = self.process_pdf(pdf_file)
            
            # Save extracted text
            if text:
                with open(output_file, 'w', encoding='utf-8') as f:
                    f.write(text)
                print(f"  ✓ Saved to {output_file.name} ({time.time() - start_time:.1f}s)")
            else:
                print(f"  ✗ Failed to extract text from {pdf_file.name}")
        
        print("\n" + "="*60)
        print("Scraping complete!")


In [ ]:
if __name__ == "__main__":
    scraper = DeepSeekOCRScraper()
    scraper.run_scraper()

In [ ]:
# Download Output Files (Optional)
# If you used Option 2 (Upload), you'll want to download the results.
from google.colab import files
import shutil

output_dir = Path(CONFIG['ocr_output_dir'])
if output_dir.exists():
    # Zip the outputs
    shutil.make_archive('/content/ocr_output', 'zip', output_dir)
    print("✅ outputs zipped!") 
    
    # Trigger download
    try:
        files.download('/content/ocr_output.zip')
    except Exception as e:
        print(f"Download failed (might be blocked by browser): {e}")
        print("You can manually search for 'ocr_output.zip' in the Files sidebar and download it.")